# The data we have

This notebook goes with section 5 of the PPMI atlas (`atlas/index.html`). It answers four questions in order: what data PPMI offers, what this project uses, what you have downloaded, and how to get the rest. It ends with the SQL interface that the other notebooks use. To download the project's whole data set in one guided pass, run `atlas.ipynb`.

PPMI shares its data through the LONI Image and Data Archive (IDA). Researchers [apply for access](https://www.ppmi-info.org/access-data-specimens/download-data), accept the data use agreement and download with their own account. The agreement forbids sharing participant-level data and applying AI tools to it, so this notebook prints file names and counts only. Keep the downloads outside the repository: they live in `~/ppmi`, or in the folder named by the `PPMI_DATA` environment variable.

In [ ]:
%reload_ext autoreload
%autoreload 2
%matplotlib inline
import sys
sys.path.append("../code")

import pandas as pd

import data
import ida
import plot

pd.set_option("display.max_rows", 200, "display.max_colwidth", 120)
catalog = data.catalog()

## 1. What PPMI offers

IDA's *Download* menu has three parts:

- **Study Data**: a few hundred CSV tables and documents in the groups below. IDA names each file after its table and the download date, for example `Participant_Status_10Nov2026.csv`.
- **Image Collections**: the scans, as DICOM or NIfTI files in zips. You search for images, add them to a collection and download the collection.
- **Genetic Data**: sequencing and genotyping files.

`code/catalog.csv` lists every kind of data that Appendix A of the PPMI Data User Guide describes, plus the imaging tables in current downloads. Rows marked "various" stand for groups of similar tables. IDA's own list is the final word, because tables are added between editions of the guide.

| Kind | One row per | Joins on |
|---|---|---|
| participant table | participant | `PATNO` |
| visit table | participant and visit | `PATNO`, `EVENT_ID` |
| log | dated record, such as a medication | `PATNO` and dates |
| imaging table | scan: metadata or a derived measure | `PATNO`, `EVENT_ID`, `IMAGEID` |
| biospecimen table | sample and test | `PATNO`, `CLINICAL_EVENT` |
| curated table | participant and visit, many sources merged | `PATNO`, `EVENT_ID` |
| images, genetic files, sensor data | file | subject and visit in the file and folder names |
| document | | |

In [ ]:
plot.heatmap(pd.crosstab(catalog.group, catalog.kind), "Kinds of data in each part of IDA", "datasets")

In [ ]:
catalog[["group", "subgroup", "dataset", "kind", "format", "keys", "notes"]]

## 2. What this project uses

The `project` column marks three sets:

- **core**: the twelve tables that `audit.ipynb` needs.
- **pipeline**: three tables the clean layer also reads when they are present: MoCA, the archived Invicro DaTscan ratios and the Xing MRI acquisition metadata.
- **recommended**: what section 7 of the guide suggests adding: the data user guide, code list, data dictionary and LEDD method, the rest of the MDS-UPDRS, the medication log, diagnosis history, genetic status, biospecimen results, the Curated Data Cut, FreeSurfer and quality-control MRI tables, and image searches.

In [ ]:
catalog.query("project != ''").sort_values(["project", "group", "dataset"])[["project", "group", "subgroup", "dataset", "format", "notes"]]

## 3. What you have

`data.PPMI()` loads everything in the data folder into one in-memory DuckDB database, and `inventory()` lines the catalogue up with your files. When a table has been downloaded more than once, the newest download is used. `rows` and `participants` count rows and distinct participants.

In [ ]:
p = data.PPMI()
inventory = p.inventory()
have = inventory.query("downloaded != ''")
have[["group", "dataset", "kind", "downloaded", "release", "rows", "participants", "columns"]]

In [ ]:
plot.counts(have.set_index("dataset").participants.dropna().astype(int).sort_values(ascending=False), "Participants in each download")

Still missing from the project's sets, and anything the loader could not read:

In [ ]:
for note in p.notes:
    print(note)
inventory.query("project != '' and downloaded == ''")[["project", "group", "subgroup", "dataset", "format"]]

## 4. Choose what to download

`wanted` starts as everything in the project's sets that you do not have yet. Edit the list with names from the `dataset` column in section 1. `where` shows where each one sits in IDA.

In [ ]:
wanted = inventory.query("project != '' and downloaded == ''").dataset.tolist()
where = catalog.set_index("dataset").loc[wanted, ["group", "subgroup", "kind", "format"]]
where

## 5. Download

**Study data and documents.** Set `open_ida = True` and run the cell: IDA's *Download Study Data* page opens in your browser. Sign in, open each group in the left-hand panel or type a dataset's name into the search box, tick the datasets in `where`, and click *Download*. Several ticked files arrive as one zip. IDA offers no programming interface for study data, so this step happens in the browser.

**Image searches.** In IDA choose *Search*, then *Advanced Search*. Pick the research group, modality and visits, and show at least Subject ID, Visit, Modality, Image ID, Description and Study Date in the results. Save them with *CSV Download*. They become the `images` table.

**Images.** Tick series in the search results and choose *Add to Collection*. Under *Data Collections*, choose *Advanced Download* and save the list of URLs it offers, then put that file's path in `url_list`. `ida.fetch` downloads every part, resumes broken transfers and unpacks the images into `images/`, where they become the `image_files` table. A small collection can come as one zip from *1-Click Download* instead.

Then run `ida.collect()`. It files what IDA saved in your Downloads folder: study CSVs and the Curated Data Cut workbook go to `study/` with IDA's names, image-search CSVs to `imaging/`, PDFs that come in a zip of study tables to `docs/`, zips are unpacked and kept in `downloads/`, and files that did not come from IDA stay where they are.

In [ ]:
open_ida = False
if open_ida:
    ida.browse("study")

In [ ]:
ida.collect()

In [ ]:
url_list = None
ida.fetch(url_list) if url_list else None

## 6. Query everything with SQL

Reload after new downloads with `p = data.PPMI()`. One connection holds two layers.

- **main** has every downloaded table exactly as IDA ships it, named after its file in lower case: `Letter_-_Number_Sequencing_10Nov2026.csv` becomes `letter_number_sequencing`. Every column is text; the macros `num(x)` and `ppmi_date(x)` turn values into numbers and dates. `PATNO` and `EVENT_ID` are trimmed so that tables join cleanly. `images` holds the image searches, `image_files` the downloaded images, `catalog` the inventory above, and `cognition` stacks the six cognitive tests into one view.
- **clean** has the pipeline in `code/ppmi_pipeline`: typed and deduplicated tables keyed by `patient_id` and `event_id`, such as `clean.patients`, `clean.visits`, `clean.cognitive_assessments`, `clean.datscan_scans` and `clean.structural_scans`, with coverage views built on them.

`schema.ipynb` maps how all of these tables relate.

In [ ]:
p = data.PPMI()
p.schema().groupby(["layer", "table"]).size().rename("columns").reset_index()

In [ ]:
p.sql("""
    SELECT COHORT_DEFINITION AS cohort, lower(ENROLL_STATUS) AS status, count(*) AS participants
    FROM participant_status GROUP BY ALL ORDER BY cohort, status
""")

In [ ]:
p.sql("SELECT * FROM clean.availability_summary")

In [ ]:
coverage = p.sql("""
    SELECT event_id,
           count(*) FILTER (has_six_tests) AS "all six tests",
           count(*) FILTER (has_datscan) AS "DaTscan",
           count(*) FILTER (has_structural) AS "grey-matter volume"
    FROM clean.visit_coverage GROUP BY event_id
""").set_index("event_id").reindex(["SC", "BL", "V04", "V06", "V08", "V10", "V12"])
plot.grouped(coverage, "Participants with each measurement, by visit")

### Outside Python

From the repository root:

```bash
python code/data.py                                    # what is loaded
python code/data.py "SELECT count(*) FROM clean.patients"
python code/data.py --export ~/ppmi/ppmi.duckdb        # both layers in one DuckDB file
duckdb ~/ppmi/ppmi.duckdb                              # query it with the DuckDB CLI or any SQL client
```

The exported file holds participant-level data. Keep it in the data folder, never in the repository.